# Actividad en clase — Autorreconocimiento de rostro con Reinforcement Learning

**Objetivo:** implementar en Python un agente de RL que, usando la **cámara incorporada**,
aprenda a reconocer **mi propio rostro** ("soy yo" / "no soy yo") a partir de la
**recompensa** que le da el usuario, sin entrenamiento supervisado clásico.

## Formulación como problema de RL (*contextual bandit*)

| Elemento | En este problema |
|---|---|
| **Estado** $s$ | vector de características del rostro detectado en el frame (PCA de la imagen recortada) |
| **Acciones** $a$ | `0 = no soy yo`, `1 = SOY YO`, `2 = no estoy seguro` (abstenerse) |
| **Recompensa** $r$ | $+1$ si acierta, $-1$ si se equivoca, $-0.1$ si se abstiene |
| **Política** | $\varepsilon$-greedy sobre $Q(s,a)$ |
| **Modelo de valor** | $Q(s,a) = w_a^\top \phi(s)$ — aproximación lineal de la función de valor |

Cada frame es un **episodio de un solo paso**: el agente ve el estado, actúa, recibe recompensa
y actualiza $Q$. Es RL real (aprende de recompensas, explora vs. explota) pero sin descuento
temporal, porque la decisión de un frame no cambia el siguiente.

La penalización de $-0.1$ por abstenerse es clave: **prefiere decir "no estoy seguro"
antes que equivocarse**, que es exactamente lo que se quiere en un sistema de identificación.

## 0. Dependencias

Se necesita `opencv-python` para la cámara y la detección de rostros.
Si no está instalado, ejecuta la celda siguiente (una sola vez).

In [ ]:
# Recomendado para el camino clásico con Haar cascades:
# !pip install "opencv-python<5"
#
# Si ya tienes OpenCV 5.x, no hace falta bajar de versión: la celda de
# percepción usa YuNet (DNN) o la región central del frame.


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

try:
    import cv2
    CV2_OK = True
    print('OpenCV', cv2.__version__)
except ImportError:
    CV2_OK = False
    print('OpenCV no disponible -> el notebook usará MODO SIMULACIÓN '
          '(instala opencv-python para usar la cámara)')

rng = np.random.default_rng(7)

TINTA, TINTA_SUAVE = '#22252a', '#6b7280'
AZUL, NARANJA, VERDE = '#2f6fd0', '#c2410c', '#166534'

# --- Configuración del problema -------------------------------------------
TAM = 48                # las caras se normalizan a 48x48 en escala de grises
N_PCA = 30              # dimensión del estado
ACCIONES = ['no soy yo', 'SOY YO', 'no estoy seguro']
R_ACIERTO, R_ERROR, R_ABSTENCION = 1.0, -1.0, -0.1

## 1. Percepción: detectar y normalizar el rostro

El detector es solo el **sensor** (encontrar *dónde* hay una cara); decidir *de quién* es
la cara es el trabajo del agente de RL. Se elige automáticamente el mejor disponible:

| Backend | Cuándo se usa |
|---|---|
| **Haar cascade** | OpenCV 4.x (`cv2.CascadeClassifier`) |
| **YuNet (DNN)** | OpenCV 5.x, si el archivo `.onnx` está descargado (`descargar_yunet()`) |
| **Región central** | sin detector: se toma el cuadro central del frame (sirve porque uno se sienta frente a la cámara) |

> **Ojo con OpenCV 5:** eliminó `CascadeClassifier` y los XML de Haar. Si prefieres el camino
> clásico, ejecuta `pip install "opencv-python<5"`; si no, usa YuNet o la región central.

In [ ]:
import os
import urllib.request

RUTA_YUNET = 'face_detection_yunet_2023mar.onnx'
URL_YUNET = ('https://github.com/opencv/opencv_zoo/raw/main/models/'
             'face_detection_yunet/face_detection_yunet_2023mar.onnx')


def descargar_yunet(ruta=RUTA_YUNET):
    """Descarga el modelo YuNet (~230 KB) para detectar rostros en OpenCV 5."""
    if not os.path.exists(ruta):
        print('Descargando YuNet...')
        urllib.request.urlretrieve(URL_YUNET, ruta)
    print('Modelo listo:', ruta)
    return ruta


def crear_detector():
    """Devuelve (backend, objeto) con el mejor detector disponible."""
    if not CV2_OK:
        return 'ninguno', None

    # 1) Haar cascade (OpenCV 4.x)
    if hasattr(cv2, 'CascadeClassifier'):
        ruta = os.path.join(cv2.data.haarcascades,
                            'haarcascade_frontalface_default.xml')
        if os.path.exists(ruta):
            casc = cv2.CascadeClassifier(ruta)
            if not casc.empty():
                return 'haar', casc

    # 2) YuNet DNN (OpenCV 5.x) — requiere descargar_yunet()
    if hasattr(cv2, 'FaceDetectorYN') and os.path.exists(RUTA_YUNET):
        det = cv2.FaceDetectorYN.create(RUTA_YUNET, '', (320, 320), 0.7, 0.3, 500)
        return 'yunet', det

    # 3) Sin detector: se usa el cuadro central del frame
    return 'centro', None


BACKEND, DETECTOR = crear_detector()
print(f'Detector de rostros: {BACKEND}')
if BACKEND == 'centro' and CV2_OK:
    print('  -> sin detector real: se usará la región central del frame.')
    print('     Ejecuta descargar_yunet() y luego: BACKEND, DETECTOR = crear_detector()')
    print('     o instala opencv-python<5 para usar Haar cascades.')


def _caja_central(frame):
    h, w = frame.shape[:2]
    lado = int(0.55 * min(h, w))
    return [(w // 2 - lado // 2, h // 2 - lado // 2, lado, lado)]


def detectar_rostros(frame_bgr):
    """Devuelve la lista de cajas (x, y, w, h) de los rostros del frame."""
    if BACKEND == 'haar':
        gris = cv2.equalizeHist(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY))
        return list(DETECTOR.detectMultiScale(gris, scaleFactor=1.15,
                                              minNeighbors=6, minSize=(90, 90)))

    if BACKEND == 'yunet':
        h, w = frame_bgr.shape[:2]
        DETECTOR.setInputSize((w, h))
        _, caras = DETECTOR.detect(frame_bgr)
        if caras is None:
            return []
        return [tuple(int(v) for v in c[:4]) for c in caras]

    return _caja_central(frame_bgr)


def recortar_normalizar(frame_bgr, caja):
    """Recorta el rostro y lo lleva a un vector 48x48 normalizado (percepción)."""
    x, y, w, h = (int(v) for v in caja)
    h_img, w_img = frame_bgr.shape[:2]
    x, y = max(0, x), max(0, y)
    w, h = min(w, w_img - x), min(h, h_img - y)
    gris = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    cara = cv2.resize(gris[y:y + h, x:x + w], (TAM, TAM),
                      interpolation=cv2.INTER_AREA)
    cara = cv2.equalizeHist(cara)                 # robustez a la iluminación
    return cara.astype(np.float32).ravel() / 255.0


## 2. Recolección de experiencia con la cámara

El agente necesita un flujo de rostros con la retroalimentación del usuario. Se capturan dos grupos:

- **`mias`**: frames con mi rostro (muévete, gira la cabeza, cambia de expresión y de luz).
- **`otras`**: frames con otro rostro — un compañero, una foto en el celular o una cara en pantalla.

> Se abre una ventana de OpenCV. **`c`** captura, **`q`** termina. Ejecuta cada celda por separado.

In [ ]:
def capturar(n_objetivo=60, etiqueta='mias', camara=0, automatico=True, pausa=0.08):
    """Captura n_objetivo rostros desde la cámara incorporada.

    automatico=True guarda cada rostro detectado; False espera la tecla 'c'.
    Devuelve un array (n, TAM*TAM) de vectores de percepción.
    """
    if not CV2_OK:
        raise RuntimeError('Se requiere opencv-python para usar la cámara')

    cap = cv2.VideoCapture(camara, cv2.CAP_DSHOW)   # CAP_DSHOW: cámara integrada en Windows
    if not cap.isOpened():
        raise RuntimeError('No se pudo abrir la cámara')

    muestras = []
    modo = 'auto' if automatico else "tecla 'c'"
    try:
        while len(muestras) < n_objetivo:
            ok, frame = cap.read()
            if not ok:
                break
            frame = cv2.flip(frame, 1)
            cajas = detectar_rostros(frame)

            for (x, y, w, h) in cajas[:1]:                     # solo el rostro principal
                cv2.rectangle(frame, (x, y), (x + w, y + h), (208, 111, 47), 2)

            cv2.putText(frame, f'[{etiqueta}] {len(muestras)}/{n_objetivo}  '
                               f"{modo} | 'q' salir",
                        (12, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            cv2.imshow('Captura de muestras', frame)
            tecla = cv2.waitKey(1) & 0xFF

            if tecla == ord('q'):
                break
            if len(cajas) and (automatico or tecla == ord('c')):
                muestras.append(recortar_normalizar(frame, cajas[0]))
                if automatico:
                    time.sleep(pausa)                          # variedad entre frames
    finally:
        cap.release()
        cv2.destroyAllWindows()
        for _ in range(4):
            cv2.waitKey(1)

    print(f'Capturadas {len(muestras)} muestras de "{etiqueta}"')
    return np.array(muestras, dtype=np.float32)

In [ ]:
# --- Modo simulación: rostros sintéticos para que el notebook corra sin cámara ---
def rostros_sinteticos(n, clase, ruido=0.10):
    """Genera 'rostros' sintéticos separables: clase 1 = yo, clase 0 = otros."""
    ys, xs = np.mgrid[0:TAM, 0:TAM] / TAM
    out = []
    for _ in range(n):
        cx, cy = 0.5 + rng.normal(0, .03), 0.5 + rng.normal(0, .03)
        ancho = (0.16 if clase == 1 else 0.24) + rng.normal(0, .015)
        cara = np.exp(-(((xs - cx) ** 2 + ((ys - cy) * (1.25 if clase == 1 else 0.9)) ** 2)
                        / (2 * ancho ** 2)))
        oy = 0.38 if clase == 1 else 0.45                      # "ojos"
        for ox in (0.36, 0.64):
            cara -= 0.6 * np.exp(-(((xs - ox) ** 2 + (ys - oy) ** 2) / (2 * 0.05 ** 2)))
        cara += rng.normal(0, ruido, cara.shape)
        cara = np.clip(cara, 0, 1)
        out.append(cara.astype(np.float32).ravel())
    return np.array(out, dtype=np.float32)


USAR_CAMARA = CV2_OK      # ponlo en False para forzar el modo simulación

if USAR_CAMARA:
    print('Mírate a la cámara y muévete un poco...')
    X_mias = capturar(60, 'mias')
    print('\nAhora muestra OTRO rostro (compañero, foto en el celular)...')
    X_otras = capturar(60, 'otras')
else:
    X_mias = rostros_sinteticos(60, clase=1)
    X_otras = rostros_sinteticos(60, clase=0)
    print('MODO SIMULACIÓN: 60 + 60 rostros sintéticos generados')

X = np.vstack([X_mias, X_otras])
Y = np.hstack([np.ones(len(X_mias), int), np.zeros(len(X_otras), int)])
print(f'Dataset de experiencia: {X.shape[0]} rostros, {X.shape[1]} píxeles cada uno')

In [ ]:
# Vista rápida de lo que "ve" el agente
fig, axes = plt.subplots(2, 8, figsize=(11, 3.1))
for k in range(8):
    axes[0, k].imshow(X_mias[k].reshape(TAM, TAM), cmap='gray')
    axes[1, k].imshow(X_otras[k].reshape(TAM, TAM), cmap='gray')
for ax in axes.ravel():
    ax.axis('off')
axes[0, 0].set_title('yo', fontsize=9, color=TINTA, loc='left')
axes[1, 0].set_title('otros', fontsize=9, color=TINTA, loc='left')
fig.suptitle('Percepción normalizada (48x48, gris, ecualizada)', fontsize=11, color=TINTA)
plt.tight_layout()
plt.show()

## 3. Representación del estado: PCA (*eigenfaces*)

2304 píxeles son demasiados para el agente. Se proyectan a 30 componentes principales,
implementadas con `numpy` (sin dependencias extra). Ese vector de 30 números **es el estado** $s$.

In [ ]:
class PCA:
    """PCA por SVD, con blanqueo, implementada a mano."""

    def fit(self, X, k):
        self.media = X.mean(axis=0)
        U, S, Vt = np.linalg.svd(X - self.media, full_matrices=False)
        self.comp = Vt[:k]
        self.escala = S[:k] / np.sqrt(len(X)) + 1e-8
        self.var = (S ** 2)[:k].sum() / (S ** 2).sum()
        return self

    def transform(self, X):
        X = np.atleast_2d(X)
        return ((X - self.media) @ self.comp.T) / self.escala


# separación entrenamiento / prueba (la prueba nunca se usa para aprender)
idx = rng.permutation(len(X))
corte = int(0.75 * len(X))
tr, te = idx[:corte], idx[corte:]

pca = PCA().fit(X[tr], N_PCA)
S_tr, S_te = pca.transform(X[tr]), pca.transform(X[te])
Y_tr, Y_te = Y[tr], Y[te]

print(f'Estado: {N_PCA} dimensiones | varianza retenida: {100 * pca.var:.1f}%')
print(f'Entrenamiento (experiencia): {len(S_tr)} | Prueba: {len(S_te)}')

fig, axes = plt.subplots(1, 6, figsize=(11, 2.2))
for k, ax in enumerate(axes):
    ax.imshow(pca.comp[k].reshape(TAM, TAM), cmap='gray')
    ax.set_title(f'PC{k + 1}', fontsize=9, color=TINTA_SUAVE)
    ax.axis('off')
fig.suptitle('Eigenfaces: las direcciones que definen el estado', fontsize=11, color=TINTA)
plt.tight_layout()
plt.show()

## 4. El agente: Q lineal + $\varepsilon$-greedy

$$Q(s,a) = w_a^\top \phi(s), \qquad \phi(s) = [\,s,\; 1\,]$$

Actualización por descenso de gradiente sobre el error de valor (aquí el retorno es la
recompensa inmediata, porque el episodio dura un paso):

$$w_a \leftarrow w_a + \frac{\alpha}{\|\phi(s)\|^2}\,\big[\, r - Q(s,a) \,\big]\,\phi(s)$$

Solo se actualiza el peso de la **acción ejecutada**: el agente nunca ve la etiqueta correcta,
solo si su decisión le dio recompensa o castigo.

In [ ]:
class AgenteRostro:
    def __init__(self, dim, n_acciones=3, alpha=0.30, eps=1.0,
                 eps_min=0.02, eps_decay=0.9985):
        self.W = np.zeros((n_acciones, dim + 1))
        self.alpha, self.eps = alpha, eps
        self.eps_min, self.eps_decay = eps_min, eps_decay
        self.n_acciones = n_acciones

    @staticmethod
    def phi(s):
        return np.append(np.ravel(s), 1.0)

    def q(self, s):
        return self.W @ self.phi(s)

    def actuar(self, s, explorar=True):
        if explorar and rng.random() < self.eps:
            return int(rng.integers(self.n_acciones))
        return int(np.argmax(self.q(s)))

    def aprender(self, s, a, r):
        """Gradiente normalizado (NLMS): el paso se divide por |phi|^2 para que
        el aprendizaje sea estable aunque el estado tenga 30 dimensiones."""
        f = self.phi(s)
        error = r - self.W[a] @ f
        self.W[a] += self.alpha * error * f / (f @ f)
        return abs(error)

    def decaer(self):
        self.eps = max(self.eps_min, self.eps * self.eps_decay)

    def confianza(self, s):
        """Margen entre la mejor acción y la segunda: qué tan seguro está."""
        q = np.sort(self.q(s))
        return float(q[-1] - q[-2])


def recompensa(accion, etiqueta_real):
    """El 'entorno' (el usuario) premia o castiga la decisión del agente."""
    if accion == 2:
        return R_ABSTENCION
    return R_ACIERTO if accion == etiqueta_real else R_ERROR


print('Agente definido:', ACCIONES)

## 5. Entrenamiento por RL — se muestran las iteraciones

Cada episodio: se muestrea un rostro de la experiencia recolectada, el agente decide,
el entorno lo premia o lo castiga, el agente ajusta $Q$.

In [ ]:
EPISODIOS = 3000
agente = AgenteRostro(N_PCA, alpha=0.30, eps=1.0, eps_decay=0.9985)

hist_r, hist_ok, hist_eps, hist_err = [], [], [], []

print(f'{"episodio":>9} | {"epsilon":>7} | {"recomp(100)":>11} | '
      f'{"acierto(100)":>12} | {"|error TD|":>10}')
print('-' * 62)

for ep in range(1, EPISODIOS + 1):
    i = int(rng.integers(len(S_tr)))
    s, y = S_tr[i], Y_tr[i]

    a = agente.actuar(s)                 # explorar / explotar
    r = recompensa(a, y)                 # el usuario da la recompensa
    err = agente.aprender(s, a, r)       # actualiza Q(s,a)
    agente.decaer()

    hist_r.append(r)
    hist_ok.append(1.0 if a == y else 0.0)
    hist_eps.append(agente.eps)
    hist_err.append(err)

    if ep <= 5 or ep % 250 == 0:
        print(f'{ep:>9} | {agente.eps:>7.3f} | {np.mean(hist_r[-100:]):>11.3f} | '
              f'{np.mean(hist_ok[-100:]):>12.3f} | {np.mean(hist_err[-100:]):>10.3f}')

print(f'\nRecompensa media de los últimos 300 episodios: {np.mean(hist_r[-300:]):.3f}')

In [ ]:
def media_movil(x, k=100):
    x = np.asarray(x, float)
    return x if len(x) < k else np.convolve(x, np.ones(k) / k, mode='valid')


fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
series = [
    (media_movil(hist_r), 'Recompensa media', 'Recompensa por episodio (media móvil 100)'),
    (media_movil(hist_ok), 'Tasa de acierto', 'Aciertos del agente (media móvil 100)'),
    (media_movil(hist_err), '|error TD|', 'Error de valor: Q converge'),
]
for ax, (y, etiqueta_y, titulo) in zip(axes, series):
    ax.plot(range(len(y)), y, color=AZUL, linewidth=2.0)
    ax.set_title(titulo, fontsize=10.5, color=TINTA)
    ax.set_xlabel('Episodio', fontsize=9, color=TINTA_SUAVE)
    ax.set_ylabel(etiqueta_y, fontsize=9, color=TINTA_SUAVE)
    ax.grid(color='#e5e7eb', linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8, colors=TINTA_SUAVE)
    for lado in ('top', 'right'):
        ax.spines[lado].set_visible(False)
axes[0].axhline(1.0, color=NARANJA, linestyle='--', linewidth=1.3)
axes[0].text(len(series[0][0]) * 0.45, 0.86, 'máximo posible = 1.0',
             fontsize=8.5, color=NARANJA)
plt.tight_layout()
plt.show()

## 6. Evaluación con la política greedy (sin exploración)

In [ ]:
def evaluar(agente, S, Y, nombre=''):
    dec = np.array([agente.actuar(s, explorar=False) for s in S])
    r = np.array([recompensa(a, y) for a, y in zip(dec, Y)])
    decididos = dec != 2
    acc = np.mean(dec[decididos] == Y[decididos]) if decididos.any() else 0.0
    print(f'{nombre:<14} recompensa media {r.mean():>6.3f} | '
          f'acierto (cuando decide) {100 * acc:>5.1f}% | '
          f'abstenciones {100 * np.mean(~decididos):>5.1f}%')
    return dec


dec_tr = evaluar(agente, S_tr, Y_tr, 'Experiencia:')
dec_te = evaluar(agente, S_te, Y_te, 'Prueba:')

# matriz de decisiones: etiqueta real x acción elegida
M = np.zeros((2, 3), int)
for a, y in zip(dec_te, Y_te):
    M[y, a] += 1

fig, ax = plt.subplots(figsize=(6.4, 2.9))
im = ax.imshow(M, cmap='Blues')
ax.set_xticks(range(3), ACCIONES, fontsize=9)
ax.set_yticks(range(2), ['real: otros', 'real: yo'], fontsize=9)
for i in range(2):
    for j in range(3):
        ax.text(j, i, M[i, j], ha='center', va='center', fontsize=11,
                color='white' if M[i, j] > M.max() / 2 else TINTA)
ax.set_title('Decisiones del agente en el conjunto de prueba', fontsize=11, color=TINTA)
ax.tick_params(length=0, colors=TINTA_SUAVE)
for lado in ax.spines.values():
    lado.set_visible(False)
plt.tight_layout()
plt.show()

## 7. Reconocimiento en vivo con la cámara + aprendizaje continuo

El agente decide en cada frame y **sigue aprendiendo de la recompensa del usuario**:

| Tecla | Efecto |
|---|---|
| **`k`** | "correcto" → recompensa $+1$ a la acción que acaba de tomar |
| **`x`** | "incorrecto" → recompensa $-1$ |
| **`q`** | salir |

Así se corrige en tiempo real: si te reconoce mal con otra luz o con gafas, presiona `x`
unas cuantas veces y verás cómo cambia su decisión.

In [ ]:
def reconocer_en_vivo(agente, pca, camara=0, aprender_en_linea=True, umbral_confianza=0.15):
    if not CV2_OK:
        raise RuntimeError('Se requiere opencv-python para usar la cámara')

    cap = cv2.VideoCapture(camara, cv2.CAP_DSHOW)
    if not cap.isOpened():
        raise RuntimeError('No se pudo abrir la cámara')

    log_r, ultimo = [], None      # ultimo = (estado, accion) para la retroalimentación
    COLORES = {0: (60, 60, 200), 1: (80, 140, 40), 2: (140, 140, 140)}   # BGR

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frame = cv2.flip(frame, 1)
            cajas = detectar_rostros(frame)

            for (x, y, w, h) in cajas[:1]:
                s = pca.transform(recortar_normalizar(frame, (x, y, w, h)))[0]
                a = agente.actuar(s, explorar=False)
                conf = agente.confianza(s)
                if conf < umbral_confianza:            # poca certeza -> se abstiene
                    a = 2
                ultimo = (s, a)

                color = COLORES[a]
                cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
                cv2.rectangle(frame, (x, y - 30), (x + w, y), color, -1)
                cv2.putText(frame, f'{ACCIONES[a]}  ({conf:.2f})', (x + 6, y - 9),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)

            r_med = np.mean(log_r[-20:]) if log_r else 0.0
            cv2.putText(frame, f"'k' correcto | 'x' incorrecto | 'q' salir   "
                               f'feedback={len(log_r)} r_media={r_med:+.2f}',
                        (12, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
            cv2.imshow('Autorreconocimiento con RL', frame)

            tecla = cv2.waitKey(1) & 0xFF
            if tecla == ord('q'):
                break
            if tecla in (ord('k'), ord('x')) and ultimo is not None:
                s, a = ultimo
                r = R_ACIERTO if tecla == ord('k') else R_ERROR
                if a == 2:
                    r = R_ABSTENCION
                if aprender_en_linea:
                    agente.aprender(s, a, r)
                log_r.append(r)
    finally:
        cap.release()
        cv2.destroyAllWindows()
        for _ in range(4):
            cv2.waitKey(1)

    print(f'Retroalimentaciones recibidas: {len(log_r)} | '
          f'recompensa media: {np.mean(log_r) if log_r else 0:.3f}')
    return log_r


if USAR_CAMARA:
    log_vivo = reconocer_en_vivo(agente, pca)
else:
    log_vivo = []
    print('MODO SIMULACIÓN: ejecuta esta celda con opencv-python instalado y cámara disponible.')

## 8. Verificación después del aprendizaje en línea

In [ ]:
_ = evaluar(agente, S_te, Y_te, 'Prueba final:')

if log_vivo:
    fig, ax = plt.subplots(figsize=(7.2, 3.2))
    acumulado = np.cumsum(log_vivo)
    ax.plot(range(1, len(acumulado) + 1), acumulado, color=VERDE, linewidth=2.0)
    ax.axhline(0, color=TINTA_SUAVE, linewidth=0.9)
    ax.set_title('Recompensa acumulada durante la sesión en vivo', fontsize=11, color=TINTA)
    ax.set_xlabel('Retroalimentación del usuario', fontsize=9, color=TINTA_SUAVE)
    ax.set_ylabel('Recompensa acumulada', fontsize=9, color=TINTA_SUAVE)
    ax.grid(color='#e5e7eb', linewidth=0.8)
    ax.set_axisbelow(True)
    for lado in ('top', 'right'):
        ax.spines[lado].set_visible(False)
    plt.tight_layout()
    plt.show()

## 9. Conclusiones y notas

- El reconocimiento se planteó como un **contextual bandit**: estado = rostro percibido,
  acciones = las tres respuestas posibles, recompensa = juicio del usuario.
  El agente **nunca ve la etiqueta**, solo si acertó o falló.
- La curva de recompensa muestra la transición **exploración → explotación**: al inicio
  $\varepsilon = 1$ (adivina) y con el decaimiento converge a la política greedy.
- La acción **"no estoy seguro"** con castigo pequeño hace que el agente aprenda a callarse
  cuando el estado es ambiguo, en vez de arriesgar un $-1$.
- El aprendizaje en línea (`k` / `x`) permite corregirlo con nueva iluminación, gafas o ángulos
  sin volver a entrenar desde cero: es la ventaja del enfoque RL frente a un clasificador fijo.

**Para experimentar:** cambia `N_PCA`, `alpha` o `eps_decay`; sube el castigo de error a $-2$
y observa cómo aumentan las abstenciones; agrega una cuarta acción ("acércate más a la cámara")
para que el agente aprenda a pedir mejores observaciones.

> **Privacidad:** las imágenes se quedan en memoria, no se guardan en disco ni se envían a
> ningún servicio. Si quieres persistir el modelo, guarda solo `agente.W`, `pca.media` y
> `pca.comp` con `np.savez`, nunca las fotos.